# ML-00 Encoding Export for XGBoost Native Categorical

- `data/preprocessing/03_ml_feature_process.ipynb`가 만든 split 포함 단일 parquet를 사용한다.
- feature 연산은 생략하고, 지정한 feature별 encoding만 적용해 `ml_00_ml_*.py` 입력 파일 세트를 생성한다.

## 생성 목표
1. `ml_exp00.parquet`의 기존 `split` 컬럼을 유지한 채 XGBoost native categorical 실험용 입력을 만든다.
2. 수치형 feature는 passthrough로 유지하고, 지정한 범주형 feature만 native categorical 입력 컬럼으로 변환한다.
3. train split 기준 category 목록과 feature type 정보를 `encoding_manifest.json`에 저장한다.
4. 사람이 검토할 train/val/test parquet, manifest, feature contract 초안을 `fb_outputs` run directory에 저장한다.

## 실행 범위

포함:
- `ml_00_fb_encoding.encode_existing_split_for_ml()` 실행
- train/val/test split별 ML 입력 parquet 생성
- native categorical feature contract 초안 생성
- category metadata를 담은 encoding manifest 저장

제외:
- 신규 graph feature 계산
- XGBoost 학습 및 그 이후의 후속 작업

## 실행 전 주의

- 원본 `data/processed/ml_features/ml_exp00.parquet`는 수정하지 않는다.
- category 목록은 train split 기준으로 고정해 val/test 정보가 encoding 기준에 섞이지 않게 한다.
- 기본 overwrite는 `False`다. 같은 설정으로 재실행하려면 `RUN_ID` 변경을 권장한다.
- 이 노트북이 만든 parquet/CSV/JSON 산출물은 검토용 후보이며 commit 대상이 아니다.

# 00_경로 및 환경설정

In [12]:
from __future__ import annotations
import sys
from pathlib import Path
import pandas as pd
from IPython.display import display

# ============================================================
# 0.1 Runtime Settings
# 실행 환경: ML 담당자 기준
# - Kernel          : Local WSL
# - Code repo       : local Git repo
# - Input features  : Google Drive Graph_AML/data/processed/ml_features
# - Output artifacts: local Git repo ml/ml-00_baseline/fb_inputs, ml_inputs
# ============================================================
SEED = 42

# ------------------------------------------------------------
# Root Paths
# ------------------------------------------------------------
# 다른 환경에서 실행할 경우 보통 이 두 경로만 수정한다.
LOCAL_REPO_ROOT = Path("/mnt/d/AML_git/Graph_AML").resolve()
DRIVE_REPO_ROOT = Path("/mnt/g/내 드라이브/Graph_AML").resolve()

# ------------------------------------------------------------
# Input Paths
# ------------------------------------------------------------
# split 포함 단일 feature parquet는 Google Drive의 processed/ml_features에서 읽는다.
ML_00_IN_DRIVE_PROCESSED_DIR = DRIVE_REPO_ROOT / "data" / "processed"
ML_00_IN_SOURCE_DATA_DIR = ML_00_IN_DRIVE_PROCESSED_DIR / "ml_features"

# ------------------------------------------------------------
# Output Paths
# ------------------------------------------------------------
# fb_inputs에는 실행계획 contract, fb_outputs에는 검토용 encoding/export 산출물을 저장한다.
ML_00_BASE_DIR = LOCAL_REPO_ROOT / "ml" / "ml-00_baseline"
ML_00_FB_INPUTS_DIR = ML_00_BASE_DIR / "fb_inputs"
ML_00_FB_OUTPUTS_DIR = ML_00_BASE_DIR / "fb_outputs"

# ------------------------------------------------------------
# Module Code Paths
# ------------------------------------------------------------
# local Git repo에서 feature_build 모듈을 import한다.
ML_00_MODULE_FB_DIR = ML_00_BASE_DIR / "feature_build"

# ============================================================
# 0.2 Path Validation
# ============================================================
def require_dir(path: Path, name: str) -> None:
    # 필수 디렉터리가 없으면 후속 export를 실행하지 않고 즉시 중단한다.
    path = Path(path).resolve()
    if not path.exists():
        raise FileNotFoundError(f"{name} not found: {path}")
    if not path.is_dir():
        raise NotADirectoryError(f"{name} is not a directory: {path}")

# 시작 단계에서 반드시 존재해야 하는 디렉터리만 검증한다.
for name, path in {
    "LOCAL_REPO_ROOT": LOCAL_REPO_ROOT,
    "DRIVE_REPO_ROOT": DRIVE_REPO_ROOT,
    "ML_00_IN_SOURCE_DATA_DIR": ML_00_IN_SOURCE_DATA_DIR,
    "ML_00_BASE_DIR": ML_00_BASE_DIR,
    "ML_00_MODULE_FB_DIR": ML_00_MODULE_FB_DIR,
}.items():
    require_dir(path, name)

# ============================================================
# 0.3 Import Path Setup
# ============================================================
# local feature_build 모듈을 우선 import한다.
IMPORT_DIRS = [str(ML_00_MODULE_FB_DIR)]
IMPORT_MODULES = ("ml_00_fb_utils", "ml_00_fb_contract", "ml_00_fb_encoding")

# 중복 경로를 제거한 뒤 맨 앞에 삽입해 import 우선순서를 고정한다.
sys.path = [path for path in sys.path if path not in IMPORT_DIRS]
sys.path[:0] = IMPORT_DIRS

# 노트북 재실행 시 수정된 local 모듈을 다시 읽도록 import cache를 제거한다.
for module_name in IMPORT_MODULES:
    sys.modules.pop(module_name, None)

import ml_00_fb_utils
from ml_00_fb_contract import (
    copy_feature_columns_to_fb_inputs,
    prepare_fb_input_contract,
)
from ml_00_fb_encoding import encode_existing_split_for_ml, load_encoding_specs
from ml_00_fb_io import parquet_columns

ml_00_fb_utils.set_seed(SEED, use_torch=True)

# ============================================================
# 0.4 Configuration Summary
# ============================================================
print("=" * 80)
print("SEED                     :", SEED)
print("-" * 80)
print("LOCAL_REPO_ROOT          :", LOCAL_REPO_ROOT)
print("DRIVE_REPO_ROOT          :", DRIVE_REPO_ROOT)
print("-" * 80)
print("ML_00_IN_SOURCE_DATA_DIR :", ML_00_IN_SOURCE_DATA_DIR)
print("ML_00_FB_INPUTS_DIR      :", ML_00_FB_INPUTS_DIR)
print("ML_00_FB_OUTPUTS_DIR     :", ML_00_FB_OUTPUTS_DIR)
print("ML_00_MODULE_FB_DIR      :", ML_00_MODULE_FB_DIR)
print("=" * 80)

SEED                     : 42
--------------------------------------------------------------------------------
LOCAL_REPO_ROOT          : /mnt/d/AML_git/Graph_AML
DRIVE_REPO_ROOT          : /mnt/g/내 드라이브/Graph_AML
--------------------------------------------------------------------------------
ML_00_IN_SOURCE_DATA_DIR : /mnt/g/내 드라이브/Graph_AML/data/processed/ml_features
ML_00_FB_INPUTS_DIR      : /mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/fb_inputs
ML_00_FB_OUTPUTS_DIR     : /mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/fb_outputs
ML_00_MODULE_FB_DIR      : /mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/feature_build


# 01_실행 설정

In [13]:
# ============================================================
# 1.1 Run identifiers
# ============================================================
# EXPORT_EXPERIMENT_ID / RUN_ID: contract와 encoding 산출물 묶음 식별자
# train_val_test native categorical 노트북과 같은 prefix를 사용한다.
EXPORT_EXPERIMENT_ID = "ml_00_test"
RUN_ID = "run_02_xgb_native_cat_15f"

ARTIFACT_PREFIX = f"{EXPORT_EXPERIMENT_ID}__{RUN_ID}"

RUN_MODE = "run_fb"
ALLOWED_RUN_MODES = {"prepare_contract", "run_fb"}
if RUN_MODE not in ALLOWED_RUN_MODES:
    raise ValueError(f"Unsupported RUN_MODE: {RUN_MODE}")

# RUN_MODE 설명
# - prepare_contract: fb_inputs에 FB 실행계획 contract를 생성/검증한다.
#
#   산출물:
#   - FB_INPUT_DIR / "ml_feature_columns.csv" Drive 원본 feature columns CSV 사본. 원본은 수정하지 않는다.
#   - FB_INPUT_CONTRACT_PATH
#     예: fb_inputs/<EXPORT_EXPERIMENT_ID>/<RUN_ID>/<ARTIFACT_PREFIX>_fb_input_feature_contract.csv 
#         run_fb 모드에서 입력될 실행계획 contract다.
#
# - run_fb:
#   검토된 fb_inputs contract 기준으로 encoding/feature export를 실행한다.
#   실행 결과 parquet, manifest, 검토용 feature contract 초안을 fb_outputs에 생성한다.
#   생성 직후 FB 책임 범위의 최소 산출물 검증을 자동 수행한다.
#
#   산출물:
#   - FB_OUTPUT_DIR / f"{ARTIFACT_PREFIX}_Xy_train.parquet"
#   - FB_OUTPUT_DIR / f"{ARTIFACT_PREFIX}_Xy_val.parquet"
#   - FB_OUTPUT_DIR / f"{ARTIFACT_PREFIX}_Xy_test.parquet"
#     후속 ML 입력 후보 split parquet다.
#
#   - FB_OUTPUT_CONTRACT_PATH
#     예: fb_outputs/<EXPORT_EXPERIMENT_ID>/<RUN_ID>/<ARTIFACT_PREFIX>_feature_contract.csv
#     사람이 검토할 feature contract 후보 초안이다.
#
#   - ENCODING_MANIFEST_PATH : 
#     train 기준 category 목록, feature type, encoding metadata를 저장한다.
#
#   - FB_OUTPUT_DIR / f"{ARTIFACT_PREFIX}_feature_types.json"
#     feature별 XGBoost type metadata다.
#
#   - FB_OUTPUT_DIR / f"{ARTIFACT_PREFIX}_category_mapping_train_only.csv"
#     train split 기준 category mapping이다.
#
#   - FB_OUTPUT_DIR / f"{ARTIFACT_PREFIX}_category_unknown_summary.csv"
#     val/test 등에서 train 기준 category에 없던 값 요약이다.
#
#   - FB_OUTPUT_DIR / f"{ARTIFACT_PREFIX}_split_summary.csv"
#     train/val/test row 및 label 분포 요약이다.

# ============================================================
# 1.2 Source input file
# ============================================================
# SOURCE_PARQUET_FILE은 split 컬럼이 포함된 단일 feature parquet다.
# FEATURE_COLUMNS_SOURCE_FILE은 Drive 원본 CSV다. 이 파일은 수정하지 않는다.
# 실제 경로는 bootstrap 셀에서 정의한 ML_00_IN_SOURCE_DATA_DIR 기준으로 만든다.
SOURCE_PARQUET_FILE = "ml_exp00.parquet"
FEATURE_COLUMNS_SOURCE_FILE = "ml_feature_columns.csv"

INPUT_SINGLE_PARQUET_PATH = ML_00_IN_SOURCE_DATA_DIR / SOURCE_PARQUET_FILE
FEATURE_COLUMNS_SOURCE_FILE_PATH = ML_00_IN_SOURCE_DATA_DIR / FEATURE_COLUMNS_SOURCE_FILE

# 기존 15개 feature 의미를 유지하되 categorical feature는 native categorical output으로 대체한다.
# pair 계열은 raw pair column이 없으므로 pair code 컬럼 자체를 categorical source로 사용한다.
SELECTED_PASSTHROUGH_FEATURES = (
    "amount__current__value",
    "amount__current__log1p",
    "amount_received__current__value",
    "amount_received__current__log1p",
    "amount__paid_recv_ratio",
    "time__row__dayofweek",
    "time__row__hour",
    "time__row__is_weekend",
)

NATIVE_CATEGORICAL_ENCODINGS = (
    {"legacy_column": "cat__from_bank__code", "column_name": "cat__from_bank__xgb_cat", "source_column": "From Bank"},
    {"legacy_column": "cat__to_bank__code", "column_name": "cat__to_bank__xgb_cat", "source_column": "To Bank"},
    {"legacy_column": "cat__bank_pair__code", "column_name": "cat__bank_pair__xgb_cat", "source_column": "cat__bank_pair__code"},
    {"legacy_column": "cat__payment_currency__code", "column_name": "cat__payment_currency__xgb_cat", "source_column": "Payment Currency"},
    {"legacy_column": "cat__receiving_currency__code", "column_name": "cat__receiving_currency__xgb_cat", "source_column": "Receiving Currency"},
    {"legacy_column": "cat__currency_pair__code", "column_name": "cat__currency_pair__xgb_cat", "source_column": "cat__currency_pair__code"},
    {"legacy_column": "cat__payment_format__code", "column_name": "cat__payment_format__xgb_cat", "source_column": "Payment Format"},
)

SELECTED_FEATURE_ORDER = (
    "amount__current__value",
    "amount__current__log1p",
    "amount_received__current__value",
    "amount_received__current__log1p",
    "amount__paid_recv_ratio",
    "cat__from_bank__xgb_cat",
    "cat__to_bank__xgb_cat",
    "cat__bank_pair__xgb_cat",
    "cat__payment_currency__xgb_cat",
    "cat__receiving_currency__xgb_cat",
    "cat__currency_pair__xgb_cat",
    "cat__payment_format__xgb_cat",
    "time__row__dayofweek",
    "time__row__hour",
    "time__row__is_weekend",
)

# ============================================================
# 1.3 Output paths
# ============================================================
# FB_INPUT_DIR: Drive 원본 CSV 사본과 fb_input contract 위치
# FB_OUTPUT_DIR: encoding export 결과와 검토용 feature contract 초안 위치
# ENCODING_MANIFEST_PATH: train 기준 category 목록과 feature type metadata 저장
# FB_OUTPUT_CONTRACT_PATH: 사람이 검토할 후보 feature contract

FB_INPUT_DIR = ML_00_FB_INPUTS_DIR / EXPORT_EXPERIMENT_ID / RUN_ID
FB_OUTPUT_DIR = ML_00_FB_OUTPUTS_DIR / EXPORT_EXPERIMENT_ID / RUN_ID
FB_INPUT_FEATURE_COLUMNS_PATH = FB_INPUT_DIR / "ml_feature_columns.csv"
FB_INPUT_NATIVE_SOURCE_PATH = FB_INPUT_DIR / f"{ARTIFACT_PREFIX}_native_categorical_feature_columns.csv"
FB_INPUT_CONTRACT_PATH = FB_INPUT_DIR / f"{ARTIFACT_PREFIX}_fb_input_feature_contract.csv"
FB_OUTPUT_CONTRACT_PATH = FB_OUTPUT_DIR / f"{ARTIFACT_PREFIX}_feature_contract.csv"

ENCODING_MANIFEST_PATH = FB_OUTPUT_DIR / f"{ARTIFACT_PREFIX}_encoding_manifest.json"

# ============================================================
# 1.4 Execution switches / overwrite policy
# ============================================================
# False이면 기존 산출물이 있을 때 중단한다. 재실행은 RUN_ID 변경을 권장한다.
OVERWRITE_CONTRACT_OUTPUTS = False
OVERWRITE_ENCODING_OUTPUTS = False

# ============================================================
# 1.5 Configuration preview
# ============================================================
print("=" * 80)
print("EXPORT_EXPERIMENT_ID   :", EXPORT_EXPERIMENT_ID)
print("RUN_ID                 :", RUN_ID)
print("ARTIFACT_PREFIX        :", ARTIFACT_PREFIX)
print("RUN_MODE               :", RUN_MODE)
print("-" * 80)
print("INPUT_SINGLE_PARQUET_PATH :", INPUT_SINGLE_PARQUET_PATH)
print("FEATURE_COLUMNS_SOURCE    :", FEATURE_COLUMNS_SOURCE_FILE_PATH)
print("-" * 80)
print("FB_INPUT_DIR              :", FB_INPUT_DIR)
print("FB_INPUT_NATIVE_SOURCE_PATH :", FB_INPUT_NATIVE_SOURCE_PATH)
print("FB_INPUT_CONTRACT_PATH    :", FB_INPUT_CONTRACT_PATH)
print("-" * 80)
print("FB_OUTPUT_DIR             :", FB_OUTPUT_DIR)
print("FB_OUTPUT_CONTRACT_PATH   :", FB_OUTPUT_CONTRACT_PATH)
print("ENCODING_MANIFEST_PATH    :", ENCODING_MANIFEST_PATH)
print("=" * 80)

EXPORT_EXPERIMENT_ID   : ml_00_test
RUN_ID                 : run_02_xgb_native_cat_15f
ARTIFACT_PREFIX        : ml_00_test__run_02_xgb_native_cat_15f
RUN_MODE               : run_fb
--------------------------------------------------------------------------------
INPUT_SINGLE_PARQUET_PATH : /mnt/g/내 드라이브/Graph_AML/data/processed/ml_features/ml_exp00.parquet
FEATURE_COLUMNS_SOURCE    : /mnt/g/내 드라이브/Graph_AML/data/processed/ml_features/ml_feature_columns.csv
--------------------------------------------------------------------------------
FB_INPUT_DIR              : /mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/fb_inputs/ml_00_test/run_02_xgb_native_cat_15f
FB_INPUT_NATIVE_SOURCE_PATH : /mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/fb_inputs/ml_00_test/run_02_xgb_native_cat_15f/ml_00_test__run_02_xgb_native_cat_15f_native_categorical_feature_columns.csv
FB_INPUT_CONTRACT_PATH    : /mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/fb_inputs/ml_00_test/run_02_xgb_native_cat_15f/ml_00_test__run_02_xgb_na

# 03_Encoding Export 실행

In [14]:
def make_native_categorical_feature_columns(
    source_path,
    output_path,
    native_encodings,
    *,
    source_data_path,
    selected_passthrough_features,
    selected_feature_order,
    overwrite=False,
):
    """
    source parquet 전체 컬럼 inventory와 native categorical 신규 row를 합쳐 contract source CSV를 만든다.

    원칙:
    - source parquet에 존재하는 모든 컬럼은 contract에 반드시 남긴다.
    - 기존 parquet 컬럼 row의 source_column/encoding/build_action 의미를 임의로 바꾸지 않는다.
    - 모델 입력 여부만 used_in_ml TRUE/FALSE로 나눈다.
    - native categorical feature는 기존 cat__*_code row를 고치는 대신 별도 output row로 추가한다.
    """
    source_path = Path(source_path).expanduser().resolve()
    output_path = Path(output_path).expanduser().resolve()
    source_data_path = Path(source_data_path).expanduser().resolve()

    if output_path.exists() and not overwrite:
        raise FileExistsError(f"native categorical feature columns source already exists: {output_path}")

    source_columns = parquet_columns(source_data_path)
    source_column_set = set(source_columns)
    selected_passthrough_set = set(selected_passthrough_features)

    table = pd.read_csv(source_path, encoding="utf-8-sig", dtype={"used_in_ml": "string"})
    required = {"column_name", "used_in_ml"}
    missing = required - set(table.columns)
    if missing:
        raise ValueError(f"source feature columns CSV is missing columns: {sorted(missing)}")

    table = table.copy()
    for column in (
        "source_column",
        "encoding",
        "build_action",
        "build_in_fb",
        "feature_group",
        "dtype",
        "leakage_risk",
        "excluded_reason",
        "selection_note",
    ):
        if column not in table.columns:
            table[column] = ""

    table["column_name"] = table["column_name"].astype(str).str.strip()
    table = table.drop_duplicates(subset=["column_name"], keep="first")
    metadata_by_column = table.set_index("column_name", drop=False)
    output_columns = list(table.columns)

    missing_selected_passthrough = sorted(selected_passthrough_set - source_column_set)
    if missing_selected_passthrough:
        raise ValueError(f"selected passthrough features are missing from source parquet: {missing_selected_passthrough}")

    native_output_columns = {spec["column_name"] for spec in native_encodings}
    duplicated_outputs = sorted(native_output_columns & source_column_set)
    if duplicated_outputs:
        raise ValueError(f"native categorical output columns already exist in source parquet: {duplicated_outputs}")

    legacy_columns = {spec["legacy_column"] for spec in native_encodings}
    missing_legacy = sorted(legacy_columns - source_column_set)
    if missing_legacy:
        raise ValueError(f"native categorical legacy/source inventory columns are missing from source parquet: {missing_legacy}")

    missing_native_sources = sorted({spec["source_column"] for spec in native_encodings} - source_column_set)
    if missing_native_sources:
        raise ValueError(f"native categorical source columns are missing from source parquet: {missing_native_sources}")

    rows_by_column = {}
    for column_name in source_columns:
        if column_name in metadata_by_column.index:
            row = metadata_by_column.loc[column_name].to_dict()
        else:
            row = {column: "" for column in output_columns}
            row["column_name"] = column_name
            row["feature_group"] = "source_parquet_inventory"

        row["column_name"] = column_name
        row["used_in_ml"] = "TRUE" if column_name in selected_passthrough_set else "FALSE"
        row["source_column"] = column_name
        row["encoding"] = "passthrough"
        row["build_action"] = "carry_forward"
        row["build_in_fb"] = "FALSE"

        replacement = next((spec for spec in native_encodings if spec["legacy_column"] == column_name), None)
        if replacement is not None:
            row["used_in_ml"] = "FALSE"
            row["excluded_reason"] = "replaced_by_xgb_native"
            row["selection_note"] = f"inventory row retained; model uses {replacement['column_name']}"

        rows_by_column[column_name] = row

    for spec in native_encodings:
        row = {column: "" for column in output_columns}
        row.update(
            {
                "column_name": spec["column_name"],
                "used_in_ml": "TRUE",
                "source_column": spec["source_column"],
                "encoding": "xgb_native",
                "build_action": "encode",
                "build_in_fb": "TRUE",
                "feature_group": "categorical_xgb_native",
                "dtype": "category",
                "leakage_risk": "low_if_train_only_category_manifest",
                "selection_note": "xgb_native encoded from raw/category source",
            }
        )
        if "target_experiment" in output_columns:
            row["target_experiment"] = EXPORT_EXPERIMENT_ID
        if "used_in_experiments" in output_columns:
            row["used_in_experiments"] = EXPORT_EXPERIMENT_ID
        rows_by_column[spec["column_name"]] = row

    missing_selected_order = [column for column in selected_feature_order if column not in rows_by_column]
    if missing_selected_order:
        raise ValueError(f"selected feature order contains unknown columns: {missing_selected_order}")

    ordered_columns = list(selected_feature_order) + [column for column in source_columns if column not in selected_feature_order]
    result = pd.DataFrame([rows_by_column[column] for column in ordered_columns], columns=output_columns)

    # 현재 run의 fb_inputs 아래에만 저장한다.
    output_path.parent.mkdir(parents=True, exist_ok=True)
    result.to_csv(output_path, index=False, encoding="utf-8-sig")
    print("native categorical feature columns source ready:", output_path)
    print("native categorical features:", [spec["column_name"] for spec in native_encodings])
    return output_path


def preview_encoding_specs(path):
    """
    fb_input contract를 EncodingSpec 목록으로 읽고, 실제 선택 feature만 출력한다.
    contract row count:
    - contract에 들어 있는 전체 row 수다.
    - used_in_ml=FALSE인 제외/이력 row도 포함된다.
    selected feature count:
    - used_in_ml=TRUE인 row 수다.
    - run_fb에서 실제 export/model input 후보로 쓰이는 feature 수다.
    """
    specs = load_encoding_specs(path)
    selected_specs = [spec for spec in specs if spec.used_in_ml]
    print("contract path:", path)
    print("contract row count:", len(specs))
    print("selected feature count:", len(selected_specs))

    # 전체 row가 아니라 실제 선택된 feature만 출력한다.
    # cat__*_xgb_cat row가 xgb_native로 보이면 native categorical contract가 맞게 생성된 것이다.
    for spec in selected_specs:
        print(spec)
    return specs

if RUN_MODE == "prepare_contract":
    # prepare_contract는 실행계획 contract를 만드는 단계다.
    # train/val/test parquet export는 아직 수행하지 않는다.
    # 1. Drive 원본 ml_feature_columns.csv를 fb_inputs로 복사한다.
    # 원본 파일은 수정하지 않는다.
    copied = copy_feature_columns_to_fb_inputs(
        FEATURE_COLUMNS_SOURCE_FILE_PATH,
        FB_INPUT_DIR,
        overwrite=OVERWRITE_CONTRACT_OUTPUTS,
    )

    # 2. native categorical 실험용 중간 CSV를 만든다.
    # 기존 cat__*_code는 제외하고, raw category source를 xgb_native row로 추가한다.
    contract_source = make_native_categorical_feature_columns(
        copied,
        FB_INPUT_NATIVE_SOURCE_PATH,
        NATIVE_CATEGORICAL_ENCODINGS,
        source_data_path=INPUT_SINGLE_PARQUET_PATH,
        selected_passthrough_features=SELECTED_PASSTHROUGH_FEATURES,
        selected_feature_order=SELECTED_FEATURE_ORDER,
        overwrite=OVERWRITE_CONTRACT_OUTPUTS,
    )

    # 3. 중간 CSV를 표준 fb_input contract로 정규화/검증한다.
    # 이 파일이 run_fb가 읽는 실행계획 파일이다.
    result = prepare_fb_input_contract(
        contract_source,
        FB_INPUT_CONTRACT_PATH,
        artifact_prefix=ARTIFACT_PREFIX,
        source_data_path=INPUT_SINGLE_PARQUET_PATH,
        overwrite=OVERWRITE_CONTRACT_OUTPUTS,
    )
    print("fb_input contract ready:", result)

    # 4. 생성된 contract에서 실제 선택 feature를 확인한다.
    ENCODING_SPECS = preview_encoding_specs(FB_INPUT_CONTRACT_PATH)

elif RUN_MODE == "run_fb":
    # run_fb는 prepare_contract가 만든 contract를 읽어 실제 export를 수행한다.
    # contract가 없으면 실행 순서가 잘못된 것이므로 명확히 중단한다.
    if not FB_INPUT_CONTRACT_PATH.exists():
        raise FileNotFoundError(
            "FB input contract not found. Run with RUN_MODE='prepare_contract' first. "
            f"path={FB_INPUT_CONTRACT_PATH}"
        )
    
    # 실행 직전 contract 내용을 다시 확인한다.
    ENCODING_SPECS = preview_encoding_specs(FB_INPUT_CONTRACT_PATH)

    # contract 기준으로 encoding/export를 실행한다.
    # xgb_native row는 train split 기준 category 목록을 사용해 category dtype과 manifest를 만든다.
    result = encode_existing_split_for_ml(
        input_path=INPUT_SINGLE_PARQUET_PATH,
        encoding_spec_path=FB_INPUT_CONTRACT_PATH,
        output_dir=FB_OUTPUT_DIR,
        artifact_prefix=ARTIFACT_PREFIX,
        overwrite=OVERWRITE_ENCODING_OUTPUTS,
    )
    
    print("encoding export completed and minimally validated")
    print("train_path             :", result.output_paths.train_path)
    print("val_path               :", result.output_paths.val_path)
    print("test_path              :", result.output_paths.test_path)
    print("feature_contract_path  :", result.output_paths.feature_contract_path)
    print("encoding_manifest_path :", result.output_paths.encoding_manifest_path)
    print("feature_count          :", len(result.feature_columns))
    print("feature_types          :", result.feature_types)

contract path: /mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/fb_inputs/ml_00_test/run_02_xgb_native_cat_15f/ml_00_test__run_02_xgb_native_cat_15f_fb_input_feature_contract.csv
contract row count: 39
selected feature count: 15
EncodingSpec(source_column='amount__current__value', output_column='amount__current__value', encoding='passthrough', used_in_ml=True)
EncodingSpec(source_column='amount__current__log1p', output_column='amount__current__log1p', encoding='passthrough', used_in_ml=True)
EncodingSpec(source_column='amount_received__current__value', output_column='amount_received__current__value', encoding='passthrough', used_in_ml=True)
EncodingSpec(source_column='amount_received__current__log1p', output_column='amount_received__current__log1p', encoding='passthrough', used_in_ml=True)
EncodingSpec(source_column='amount__paid_recv_ratio', output_column='amount__paid_recv_ratio', encoding='passthrough', used_in_ml=True)
EncodingSpec(source_column='From Bank', output_column='cat__from_bank